In [91]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
import joblib



**Load the Data**

In [92]:
df = pd.read_csv('Data/data.csv')
df.sample()

,Title,Year,Genre,Description
109,How to Train Your Dragon,2010,"Animation, Action, Adventure","Long ago up North on the Island of Berk, the y..."


**Encode The Title into numeric classes and name the column Label**

In [93]:
label_encoder = LabelEncoder()
df['Label'] = label_encoder.fit_transform(df['Title'])


#Backup label encoding
joblib.dump(label_encoder, 'label_encoder.pkl')
df.sample()

,Title,Year,Genre,Description,Label
363,The Diving Bell and the Butterfly,2007,"Biography, Drama",Forty-three year old Elle magazine editor Jean...,316


**Load the sentence transformer and transform description column into embeddings**

In [94]:
sentence_transforming_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = sentence_transforming_model.encode(df['Description'].tolist(), show_progress_bar=True)
#Save the embeddings
joblib.dump(embeddings, 'movie_embeddings.pkl')

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

['movie_embeddings.pkl']

In [95]:
df.to_csv('movie_data_encoded.csv', index=False)

In [96]:
def find_similar_movies_sklearn(description, top_k=3):
    # Encode input description
    input_embedding = sentence_transforming_model.encode([description])
    
    # Calculate cosine similarities (scikit-learn version)
    similarities = cosine_similarity(input_embedding.reshape(1, -1), embeddings)[0]
    
    # Get top k results
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    top_scores = similarities[top_indices]
    
    # Build result DataFrame safely
    result = df.iloc[top_indices].copy()
    result["Score"] = top_scores

    return result


In [ ]:

# Example usage
test_desc = "Biblo the hobbit is hired to accompany a wizard and a group of dwarves to reclaim their home Erabor"
predicted_data_frame = find_similar_movies_sklearn(test_desc, 3)
predicted_data_frame

,Title,Year,Genre,Description,Label,Score
97,The Hobbit: An Unexpected Journey,2012,"Adventure, Fantasy",Bilbo Baggins is swept into a quest to reclaim...,325,0.520943
172,The Lord of the Rings: The Fellowship of the Ring,2001,"Adventure, Drama, Fantasy",An ancient Ring thought lost for centuries has...,342,0.487549
168,The Lord of the Rings: The Two Towers,2002,"Adventure, Drama, Fantasy",The continuing quest of Frodo and the Fellowsh...,343,0.437004
165,Harry Potter and the Chamber of Secrets,2002,"Adventure, Family, Fantasy",Forced to spend his summer holidays with his m...,121,0.272686
90,Django Unchained,2012,"Comedy, Drama, Western","In 1858, a bounty-hunter named King Schultz se...",85,0.261977
171,Shrek,2001,"Animation, Adventure, Comedy",When a green ogre named Shrek discovers his sw...,256,0.253960
0,Snow White,2025,"Adventure, Family, Fantasy",A live-action re-imaging of Walt Disney's 1937...,268,0.251334
267,The Ballad of Buster Scruggs,2018,"Comedy, Drama, Musical",Six tales of life and violence in the Old West...,294,0.247879
109,How to Train Your Dragon,2010,"Animation, Action, Adventure","Long ago up North on the Island of Berk, the y...",127,0.235206
48,It (2017),2017,"Comedy, Talk-Show",Much like how Pennywise returns every 27 years...,143,0.232850
